In [1]:
import torch
import random

In [2]:
words = ''.join(open('names.txt')).splitlines()

In [3]:
alpha = sorted(list(set(''.join(words).lower())))
stoi = {c: i+1 for i, c in enumerate(list(alpha))}
A = len(alpha) + 1
stoi['.'] = 0
itos = {v: k for k, v in stoi.items()}

In [4]:
random.shuffle(words)

In [5]:
N = torch.zeros((A * A ,A), dtype = torch.int32)

In [6]:
for w in words:
    chars = ['.', '.'] + list(w.lower()) + ['.']
    for ch1, ch2, ch3 in zip(chars, chars[1:], chars[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        N[ix1 * A + ix2, ix3] += 1

In [7]:
P = (N + 1).float()
P = P / P.sum(1, keepdim = True)

In [8]:
gen = torch.Generator().manual_seed(42)
for i in range(10):
    ix1 = 0
    ix2 = 0
    name = ''
    while True:
        ix1, ix2 = ix2, torch.multinomial(P[ix1 * A + ix2], num_samples=1, replacement = True, generator = gen).item()
        if ix2 == 0:
            break
        name += itos[ix2]
    print(name)

anuee
nvtps
marian
dante
na
silayley
kemah
lucin
epiccaleen
dmzi


In [9]:
smnll = 0
n = 0
for w in words:
    chars = ['.', '.'] + list(w.lower()) + ['.']
    for ch1, ch2, ch3 in zip(chars, chars[1:], chars[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        smnll += -P[ix1 * A + ix2, ix3].log().item()
        n += 1
print(smnll/n)

2.212582425863173


In [10]:
xs = []
ys = []
for w in words:
    chars = ['.', '.'] + list(w.lower()) + ['.']
    for ch1, ch2, ch3 in zip(chars, chars[1:], chars[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1 * A + ix2)
        ys.append(ix3)
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [11]:
W = torch.randn((A * A, A), generator = gen, requires_grad=True)

In [12]:
epochs = 10000
lr = 50
for epoch in range(epochs + 1):
    logits = W[xs] # without one_hot encoding, simply plug out needed row of W as the logits
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim = True)
    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    if epoch % 200 == 0:
        lr*=0.9
        print(f'Loss: {loss.item()}')

    W.grad = None

    loss.backward()

    W.data += -lr * W.grad

Loss: 3.786858558654785
Loss: 2.392333507537842
Loss: 2.3102638721466064
Loss: 2.278550148010254
Loss: 2.2616310119628906
Loss: 2.2511253356933594
Loss: 2.243990182876587
Loss: 2.238846778869629
Loss: 2.234978675842285
Loss: 2.2319765090942383
Loss: 2.2295899391174316
Loss: 2.227656126022339
Loss: 2.226065158843994
Loss: 2.2247400283813477
Loss: 2.2236249446868896
Loss: 2.2226781845092773
Loss: 2.2218687534332275
Loss: 2.2211720943450928
Loss: 2.220569610595703
Loss: 2.220045566558838
Loss: 2.2195885181427
Loss: 2.2191882133483887
Loss: 2.218836545944214
Loss: 2.21852707862854
Loss: 2.2182536125183105
Loss: 2.2180116176605225
Loss: 2.21779727935791
Loss: 2.217606782913208
Loss: 2.217437505722046
Loss: 2.2172868251800537
Loss: 2.2171525955200195
Loss: 2.2170329093933105
Loss: 2.216925859451294
Loss: 2.216830253601074
Loss: 2.2167446613311768
Loss: 2.216667890548706
Loss: 2.216599464416504
Loss: 2.216538190841675
Loss: 2.2164828777313232
Loss: 2.2164337635040283
Loss: 2.2163894176483154


In [14]:
loss.item()

2.216132879257202